# Options Pricing & Risk Engine — Interactive Demo

This notebook demonstrates the C++17 options library exposed via pybind11.

**Topics covered:**
1. Black-Scholes European pricing
2. Analytical Greeks
3. Monte Carlo pricing (pseudo-random + antithetic + Sobol)
4. Implied volatility inversion (Newton-Raphson)
5. Volatility surface construction & interpolation
6. American options via CRR binomial tree
7. Visualisations (Greeks, vol surface, P&L scenarios, MC convergence)

In [1]:
import sys, os
# Add the build directory to path if the extension is built there
sys.path.insert(0, os.path.join('..', 'build'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

try:
    import options_py as op
    print(f'C++ extension loaded.')
except ImportError as e:
    print(f'Extension not found ({e}). Run: cmake .. -DBUILD_PYTHON=ON && cmake --build . --config Release')
    op = None

Extension not found (No module named 'options_py'). Run: cmake .. -DBUILD_PYTHON=ON && cmake --build . --config Release


## 1. Black-Scholes European Pricing

In [ ]:
if op:
    S, K, r, T, sigma, q = 100.0, 100.0, 0.05, 1.0, 0.20, 0.02

    call = op.bs_call(S, K, r, T, sigma, q)
    put  = op.bs_put (S, K, r, T, sigma, q)
    pcp  = S*np.exp(-q*T) - K*np.exp(-r*T)   # put-call parity RHS

    print(f'ATM Call  : {call:.6f}')
    print(f'ATM Put   : {put:.6f}')
    print(f'C - P     : {call - put:.6f}')
    print(f'PCP check : {pcp:.6f}  (should match C - P)')
    print(f'|Error|   : {abs((call - put) - pcp):.2e}')

## 2. Analytical Greeks

In [ ]:
if op:
    g = op.analytical_greeks(S, K, r, T, sigma, q, op.CALL)
    gn = op.numerical_greeks(S, K, r, T, sigma, q, op.CALL)

    rows = [('delta', g.delta, gn.delta),
            ('gamma', g.gamma, gn.gamma),
            ('vega',  g.vega,  gn.vega),
            ('theta', g.theta, gn.theta),
            ('rho',   g.rho,   gn.rho)]

    print(f'{"Greek":<8} {"Analytical":>14} {"Numerical":>14} {"Rel Error":>12}')
    print('-' * 52)
    for name, a, n in rows:
        rel = abs(a - n) / (abs(a) + 1e-12)
        print(f'{name:<8} {a:>14.8f} {n:>14.8f} {rel:>12.2e}')

## 3. Monte Carlo Pricing — Variance Reduction Comparison

In [ ]:
if op:
    bs_ref = op.bs_call(S, K, r, T, sigma, q)
    N = 100_000

    configs = [
        ('Standard MC',          {'antithetic': False, 'use_sobol': False, 'control_vrt': False}),
        ('Antithetic variates',  {'antithetic': True,  'use_sobol': False, 'control_vrt': False}),
        ('Sobol + antithetic',   {'antithetic': True,  'use_sobol': True,  'control_vrt': False}),
        ('Antithetic + CV',      {'antithetic': True,  'use_sobol': False, 'control_vrt': True}),
    ]

    print(f'{"Method":<24} {"Price":>10} {"Std Err":>10} {"Error vs BS":>12} {"ms":>8}')
    print('-' * 70)
    for name, kwargs in configs:
        cfg = op.MCConfig()
        cfg.num_paths = N
        for k, v in kwargs.items():
            setattr(cfg, k, v)
        eng = op.MonteCarloEngine(cfg)
        res = eng.price_european(S, K, r, T, sigma, q, op.CALL)
        print(f'{name:<24} {res.price:>10.6f} {res.std_error:>10.6f}'
              f' {abs(res.price-bs_ref):>12.6f} {res.elapsed_ms:>8.1f}')

## 4. Implied Volatility Inversion

In [ ]:
if op:
    true_vols = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.60]
    print(f'{"True σ":>8} {"IV solved":>10} {"Error":>10} {"Iters":>6}')
    print('-' * 40)
    for sv in true_vols:
        mkt = op.bs_call(S, K, r, T, sv, q)
        iv  = op.implied_vol(mkt, S, K, r, T, q, op.CALL)
        print(f'{sv:>8.2%} {iv.implied_vol:>10.6f} {abs(iv.implied_vol-sv):>10.2e} {iv.iterations:>6d}')

## 5. Volatility Surface

In [ ]:
if op:
    expiries = [0.25, 0.5, 1.0, 2.0]
    strikes  = [80., 90., 95., 100., 105., 110., 120.]

    # Generate synthetic market prices with a skew
    def smile_vol(K_val, T_val):
        return 0.20 - 0.10*(K_val/S - 1.0)/np.sqrt(T_val)

    market_prices = [
        [op.bs_call(S, K_val, r, T_val, smile_vol(K_val, T_val), q)
         for K_val in strikes]
        for T_val in expiries
    ]

    surf = op.build_vol_surface(S, r, q, expiries, strikes, market_prices, op.CALL)

    print('Vol surface (rows=expiry, cols=strike):')
    print(f'{"":>6}', '  '.join(f'{k:>7.0f}' for k in strikes))
    for i, T_val in enumerate(expiries):
        row = '  '.join(f'{v:>7.4f}' for v in surf.vols[i])
        print(f'T={T_val:<4}  {row}')

    # Interpolate at intermediate point
    iv_interp = surf.interpolate(97.5, 0.75)
    print(f'\nInterpolated IV at K=97.5, T=0.75y: {iv_interp:.4f}')

## 6. American Option — CRR Binomial Tree

In [ ]:
if op:
    print('American vs European put — early exercise premium')
    print(f'{"K":>6} {"Euro put":>10} {"Amer put":>10} {"Premium":>10}')
    print('-' * 42)
    for K_val in [90., 95., 100., 105., 110.]:
        euro = op.crr_tree(S, K_val, r, T, sigma, q, op.PUT, op.EUROPEAN, 500)
        amer = op.crr_tree(S, K_val, r, T, sigma, q, op.PUT, op.AMERICAN, 500)
        prem = amer.price - euro.price
        print(f'{K_val:>6.0f} {euro.price:>10.6f} {amer.price:>10.6f} {prem:>10.6f}')

    print('\nTree vs B-S European call (convergence):')
    bs_call_ref = op.bs_call(S, K, r, T, sigma, q)
    print(f'{"Steps":>7} {"Tree price":>12} {"B-S price":>12} {"Error":>10}')
    for n in [10, 50, 100, 200, 500, 1000]:
        tree = op.crr_tree(S, K, r, T, sigma, q, op.CALL, op.EUROPEAN, n)
        print(f'{n:>7} {tree.price:>12.7f} {bs_call_ref:>12.7f} {abs(tree.price-bs_call_ref):>10.2e}')

## 7. Visualisations

In [ ]:
from viz import plot_greeks, plot_vol_surface, plot_pnl_scenarios, plot_mc_convergence

fig = plot_greeks(K=100, r=0.05, T=1.0, sigma=0.20, q=0.02)
plt.show()

In [ ]:
fig = plot_vol_surface(S=100, r=0.05, q=0.02, base_vol=0.20, skew=-0.10)
plt.show()

In [ ]:
fig = plot_pnl_scenarios(S=100, K=100, r=0.05, T=1.0, sigma=0.20)
plt.show()

In [ ]:
if op:
    fig = plot_mc_convergence(S=100, K=100, r=0.05, T=1.0, sigma=0.20)
    plt.show()